# Whisper-Large-v3 Fine-Tuning for Moroccan Darija ASR

**Companion to**: `qwen2audio_darija_finetune_v2.ipynb`

This notebook implements LoRA fine-tuning of `openai/whisper-large-v3` on Moroccan Darija using the **same controlled methodology** as the Qwen2-Audio fine-tuning, enabling a fair comparison between end-to-end audio-LLMs (Qwen2-Audio) and cascaded ASR (Whisper) architectures in Phase 1 of our PFE.

## What is identical to the Qwen2-Audio fine-tuning

- Dataset: `atlasia/MoulSot-Full`, config `100-gt-2.5`, publisher's official train/test split
- Validation split: 2000 samples carved from publisher's train with `seed=42`
- Periodic eval: scale-tuned subsets (200 utt at 10h, 300 at 20h, 500 at 30h, 1000 at 80h)
- Casablanca normalization (Talafha et al. 2024): diacritics + hamza/madda→alif + Eastern→Western digits + lowercase Latin + punctuation strip
- Code-switching detection: reference has both Arabic (U+0600–06FF) and Latin characters
- Bootstrap CIs: 1000 paired resamples, 95% percentile
- Evaluation partitioning: full / code_switched / monolingual
- OOD eval: 500 samples from UBC-NLP/Casablanca Morocco
- LoRA: rank=32, alpha=32, dropout=0.1, rsLoRA=True, target_modules=`all-linear`
- Effective batch size: 16 (per-device=4, grad-accum=4)
- Marco-ASR adaptive LR with WER-driven adjustment + early stopping (patience=3, min_delta=0.5pp)
- Auto-resume from latest checkpoint, disk-space guard, auto-fallback wrapper
- save_steps=200, save_total_limit=3, save_safetensors=True
- PyTorch 2.6 `weights_only=False` patch for HF Trainer's rng_state.pth

## What differs (architectural necessity)

- **Marco-ASR Algorithm 2** instead of Algorithm 1 (Ni et al. 2025, §2.1.3): separate adaptive LRs for encoder and decoder
  - Encoder LR range: [1e-7, 1e-5] — smaller (encoder is already strong from Whisper pretraining)
  - Decoder LR range: [1e-6, 1e-4] — larger (decoder must learn the Darija output distribution)
- **No boilerplate stripping**: Whisper produces raw transcripts, not chat-style responses
- **No "Darija prompt" variant**: Whisper conditions on the `<|ar|>` language token, not natural-language text prompts
- **Custom AdamW with named param groups**: required so Algorithm 2 can set encoder/decoder LRs independently and survive optimizer save/resume
- **Whisper-specific data collator**: tokenizes transcriptions with language/task prefix tokens, removes the leading `<|sot|>` (re-added by `shift_tokens_right` in the forward pass)
- **PEFT task_type=SEQ_2_SEQ_LM** instead of CAUSAL_LM
- **Architecture**: 635M-param encoder + 907M-param decoder = 1.54B total (vs Qwen2-Audio's 8.5B = encoder + adapter + LLM)

## Run plan

This notebook runs the same scale ladder as Qwen2-Audio: 10h first, then 20h, with patience-based early-stopping at each scale.

## References

- Ni et al. 2025, arXiv:2512.22165 — Marco-ASR framework (Algorithm 2 for encoder-decoder)
- Radford et al. 2023 — Robust speech recognition via large-scale weak supervision (Whisper)
- Talafha et al. 2024, arXiv:2410.04527 — Casablanca corpus and normalization protocol
- Hu et al. 2021 — LoRA: Low-Rank Adaptation of LLMs
- Kalajdzievski 2023 — rank-stabilized LoRA (rsLoRA)


In [1]:
# === CELL 1: Configuration ===
# All knobs in one place. Edit here, then run top-to-bottom.

import os

# --- Mode ---
# fine_tune     : full pipeline (WER₀ → initial LRs → fine-tune → final eval)
# wer0_only     : only compute WER₀ and print initial LRs
# evaluate_only : load a pre-trained LoRA adapter and evaluate on test set
MODE = "fine_tune"

# --- Data ---
DATA_SCALE_HOURS = 10                           # 10, 20, 30, or 80
MOULSOT_DATASET_ID = "atlasia/MoulSot-Full"
MOULSOT_CONFIG = "100-gt-2.5"                   # 80h "golden" subset; publisher's official train/test split
VAL_SET_SIZE = 2000                              # carved from publisher's train (seed=42); used for WER₀ + periodic eval

CASABLANCA_DATASET_ID = "UBC-NLP/Casablanca"
CASABLANCA_SUBSET = "Morocco"                    # for OOD eval slice
CASABLANCA_OOD_MAX_SAMPLES = 500                 # cap OOD eval set size

# --- Whisper conditioning ---
# Whisper has no Darija language token; <|ar|> is the closest (standard practice in dialect Arabic ASR).
LANGUAGE = "arabic"
TASK = "transcribe"
MAX_GEN_TOKENS = 225                             # Whisper-LV3 default for ASR (no timestamps)

# --- Model ---
MODEL_ID = "openai/whisper-large-v3"
DEVICE = "cuda"

# --- Robustness / resume ---
RESUME_FROM_CHECKPOINT = True

# --- LoRA (locked per methodology, identical to Qwen2-Audio FT) ---
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
USE_RSLORA = True
LORA_BIAS = "none"

# --- Training (effective batch size 16 = 4 × 4, identical to Qwen2-Audio) ---
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_TRAIN_EPOCHS = 3                             # Marco-ASR will usually early-stop before 3
WEIGHT_DECAY = 0.01
MAX_AUDIO_DURATION_SEC = 30.0                    # Whisper encoder hard limit; longer audio filtered out

# --- Marco-ASR Algorithm 2 (Ni et al. 2025, §2.1.3) ---
# Two independent LR ranges. Encoder range is one decade lower than decoder:
#   - Encoder: well-pretrained audio representation extractor → small LR suffices
#   - Decoder: must learn Darija output distribution → larger LR
USE_MARCO_ASR = True
MARCO_LR_ENC_MIN = 1e-7
MARCO_LR_ENC_MAX = 1e-5
MARCO_LR_DEC_MIN = 1e-6
MARCO_LR_DEC_MAX = 1e-4

# Early-stopping (same as Qwen2-Audio for direct comparability)
MARCO_PATIENCE = 3
MARCO_MIN_DELTA_PCT = 0.5

# Scale-tuned eval subset + eval frequency (statistical motivation in Qwen2-Audio notebook)
MARCO_EVAL_SUBSET_BY_SCALE = {10: 200, 20: 300, 30: 500, 80: 1000}
MARCO_EVAL_STEPS_BY_SCALE = {10: 100, 20: 150, 30: 200, 80: 500}
MARCO_EVAL_SUBSET_SIZE = MARCO_EVAL_SUBSET_BY_SCALE.get(DATA_SCALE_HOURS, 200)
MARCO_EVAL_STEPS = MARCO_EVAL_STEPS_BY_SCALE.get(DATA_SCALE_HOURS, 100)
MARCO_EVAL_BATCH_SIZE = 4                        # Whisper encoder OOMs at batch≥8 with 30s audio

# --- Output paths (whisper_ prefix distinguishes from Qwen2-Audio outputs) ---
OUTPUT_DIR = "/workspace/outputs"
BEST_CKPT_DIR = f"{OUTPUT_DIR}/best_marco_asr_whisper_{DATA_SCALE_HOURS}h"
LOGS_DIR = f"{OUTPUT_DIR}/logs"
RESULTS_DIR = f"{OUTPUT_DIR}/results"
TRAINER_OUTPUT_DIR = f"{OUTPUT_DIR}/trainer_whisper_{DATA_SCALE_HOURS}h"

# --- HF Hub (optional push) ---
HUB_MODEL_ID = None  # e.g. "username/whisper-lv3-darija-10h"

# --- Evaluate-only mode ---
EVAL_ADAPTER_PATH = BEST_CKPT_DIR

# --- Bootstrap ---
BOOTSTRAP_RESAMPLES = 1000
BOOTSTRAP_CI = 0.95

# --- Misc ---
SEED = 42
DEBUG = False

# Create output directories
for d in [OUTPUT_DIR, BEST_CKPT_DIR, LOGS_DIR, RESULTS_DIR, TRAINER_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded:")
print(f"  Mode:                 {MODE}")
print(f"  Model:                {MODEL_ID}")
print(f"  Data scale:           {DATA_SCALE_HOURS}h")
print(f"  Whisper conditioning: language={LANGUAGE!r} (=<|ar|>), task={TASK!r}")
print(f"  LoRA:                 r={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, rsLoRA={USE_RSLORA}")
print(f"  Per-device batch:     {PER_DEVICE_BATCH_SIZE}")
print(f"  Grad accumulation:    {GRADIENT_ACCUMULATION_STEPS}  (effective batch = {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"  Marco-ASR Algorithm:  2 (encoder + decoder separate LRs)")
print(f"    Encoder LR range:   [{MARCO_LR_ENC_MIN:.0e}, {MARCO_LR_ENC_MAX:.0e}]")
print(f"    Decoder LR range:   [{MARCO_LR_DEC_MIN:.0e}, {MARCO_LR_DEC_MAX:.0e}]")
print(f"    Patience:           {MARCO_PATIENCE}  (min_delta = {MARCO_MIN_DELTA_PCT} pp)")
print(f"    Eval subset:        {MARCO_EVAL_SUBSET_SIZE} utt every {MARCO_EVAL_STEPS} steps")
print(f"  Output dir:           {OUTPUT_DIR}")
print(f"  Trainer dir:          {TRAINER_OUTPUT_DIR}")
print(f"  Best checkpoint dir:  {BEST_CKPT_DIR}")
print(f"  Auto-resume:          {RESUME_FROM_CHECKPOINT}")


Configuration loaded:
  Mode:                 fine_tune
  Model:                openai/whisper-large-v3
  Data scale:           10h
  Whisper conditioning: language='arabic' (=<|ar|>), task='transcribe'
  LoRA:                 r=32, alpha=32, dropout=0.1, rsLoRA=True
  Per-device batch:     4
  Grad accumulation:    4  (effective batch = 16)
  Marco-ASR Algorithm:  2 (encoder + decoder separate LRs)
    Encoder LR range:   [1e-07, 1e-05]
    Decoder LR range:   [1e-06, 1e-04]
    Patience:           3  (min_delta = 0.5 pp)
    Eval subset:        200 utt every 100 steps
  Output dir:           /workspace/outputs
  Trainer dir:          /workspace/outputs/trainer_whisper_10h
  Best checkpoint dir:  /workspace/outputs/best_marco_asr_whisper_10h
  Auto-resume:          True


## 1. Environment setup

In [1]:
# === CELL 3a: Install dependencies ===
import subprocess
import sys

packages = [
    "matplotlib",
    "scipy",
    "safetensors",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

!python -m pip install --upgrade pip -q
!pip install transformers==4.46.3
!pip install torchcodec soundfile
!pip install -qU accelerate datasets bitsandbytes hf_transfer \
     tensorboard librosa jiwer regex
!pip install peft==0.13.2
!pip install "datasets<3.0" soundfile
!apt-get update -qq && apt-get install -y -qq libsndfile1 > /dev/null 2>&1
print("Environment ready.")


  Using cached datasets-2.21.0-py3-none-any.whl.metadata (21 kB)
Using cached datasets-2.21.0-py3-none-any.whl (527 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
Environment ready.


In [2]:
# === CELL 3b: Imports ===
import os
import re
import gc
import json
import time
import random
import functools
import shutil
import unicodedata
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

import numpy as np
import torch
from torch.utils.data import Subset

from datasets import load_dataset, Audio, Dataset, concatenate_datasets
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
import jiwer
from jiwer import wer as jiwer_wer, cer as jiwer_cer
import matplotlib.pyplot as plt
import librosa

# PyTorch 2.6 compat patch: HF Trainer's rng_state.pth contains numpy state objects
# that fail under PyTorch 2.6's default weights_only=True. Patch torch.load to keep
# weights_only=False (matches behavior of older PyTorch versions).
if not getattr(torch.load, "_patched_for_hf_trainer", False):
    _original_torch_load = torch.load

    @functools.wraps(_original_torch_load)
    def _patched_torch_load(f, *args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _original_torch_load(f, *args, **kwargs)

    _patched_torch_load._patched_for_hf_trainer = True
    torch.load = _patched_torch_load
    print("✓ Patched torch.load to default weights_only=False (PyTorch 2.6+ compat)")

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


✓ Patched torch.load to default weights_only=False (PyTorch 2.6+ compat)
PyTorch 2.8.0+cu128, CUDA available: True
  Device: NVIDIA A40
  VRAM:   47.7 GB


In [3]:
# === CELL 3c: HF Hub login ===
# Set HF_TOKEN as an env var on RunPod, OR call login() interactively.
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HF Hub via HF_TOKEN env var.")
else:
    print("HF_TOKEN not set. Call login() interactively if needed:")
    print("  >>> from huggingface_hub import login; login()")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HF Hub via HF_TOKEN env var.


## 2. Model and processor loading

In [4]:
# === CELL 4: Load Whisper-LV3 processor and model ===
print(f"Loading {MODEL_ID}...")
t0 = time.time()

# Processor: feature extractor (log-mel) + tokenizer (BPE)
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
# Make sure the tokenizer prepends language + task + notimestamps tokens to labels
processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task=TASK)

# Model in bf16, SDPA attention (FA2 caused issues in Qwen2-Audio, sticking with SDPA for consistency)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)
model = model.to(DEVICE)

# Clear default forced_decoder_ids and suppress_tokens for training.
# These are restored ONLY at generation time inside our inference function.
# Rationale: during training the labels already carry the language/task tokens via
# tokenizer.set_prefix_tokens; we don't need the generate-time machinery to interfere.
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

# Pre-compute the (position, token_id) list we'll pass to .generate() at inference time
FORCED_DECODER_IDS = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
DECODER_START_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")

n_total = sum(p.numel() for p in model.parameters())
n_enc = sum(p.numel() for p in model.model.encoder.parameters())
n_dec = sum(p.numel() for p in model.model.decoder.parameters())

print(f"Loaded in {time.time()-t0:.1f}s")
print(f"  Total params:         {n_total/1e6:>8.1f}M")
print(f"  Encoder params:       {n_enc/1e6:>8.1f}M")
print(f"  Decoder params:       {n_dec/1e6:>8.1f}M  (incl. proj_out + embeddings)")
print(f"  Forced decoder IDs:   {FORCED_DECODER_IDS}")
print(f"  Decoder start token:  {DECODER_START_TOKEN_ID} (<|startoftranscript|>)")
print(f"  Device:               {next(model.parameters()).device}")
print(f"  Dtype:                {next(model.parameters()).dtype}")


Loading openai/whisper-large-v3...
Loaded in 48.5s
  Total params:           1543.5M
  Encoder params:          637.0M
  Decoder params:          906.5M  (incl. proj_out + embeddings)
  Forced decoder IDs:   [(1, 50272), (2, 50360), (3, 50364)]
  Decoder start token:  50258 (<|startoftranscript|>)
  Device:               cuda:0
  Dtype:                torch.bfloat16


## 3. Dataset loading

In [5]:
# === CELL 5: Load MoulSot-Full ===
print(f"Loading {MOULSOT_DATASET_ID} (config={MOULSOT_CONFIG})...")
t0 = time.time()
moulsot = load_dataset(MOULSOT_DATASET_ID, MOULSOT_CONFIG)
print(f"  loaded in {time.time()-t0:.1f}s")
print(f"  splits: {list(moulsot.keys())}")
for split in moulsot:
    print(f"    {split}: {len(moulsot[split])} samples")
print(f"  columns: {moulsot['train'].column_names}")


def _find_text_col(ds):
    """Find the transcription column (varies across datasets)."""
    for c in ["transcription", "text", "sentence", "transcript"]:
        if c in ds.column_names:
            return c
    raise ValueError(f"No transcription column found in {ds.column_names}")


TEXT_COL = _find_text_col(moulsot["train"])
print(f"  transcription column: '{TEXT_COL}'")

if TEXT_COL != "transcription":
    moulsot = moulsot.rename_columns({TEXT_COL: "transcription"})

# Cast audio to 16kHz (Whisper's expected sampling rate)
moulsot = moulsot.cast_column("audio", Audio(sampling_rate=16000))

# Filter out:
#   - audio > MAX_AUDIO_DURATION_SEC (Whisper encoder hard limit)
#   - audio < 0.1s (likely corrupt)
#   - empty / non-string transcriptions
def _is_valid(example):
    audio = example["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    transcription = example["transcription"]
    if not isinstance(transcription, str) or not transcription.strip():
        return False
    return 0.1 <= duration <= MAX_AUDIO_DURATION_SEC

print(f"Filtering audio > {MAX_AUDIO_DURATION_SEC}s + empty transcriptions...")
t0 = time.time()
moulsot_filt = moulsot.filter(_is_valid, num_proc=4)
print(f"  train: {len(moulsot['train'])} → {len(moulsot_filt['train'])}")
print(f"  test:  {len(moulsot['test'])} → {len(moulsot_filt['test'])}")
print(f"  filtering done in {time.time()-t0:.1f}s")


Loading atlasia/MoulSot-Full (config=100-gt-2.5)...


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

  loaded in 3.8s
  splits: ['train', 'test']
    train: 79857 samples
    test: 1993 samples
  columns: ['id', 'audio', 'sample_rate', 'n_channels', 'pesq_hyp', 'stoi_hyp', 'si_sdr_hyp', 'text', 'channel', 'duration']
  transcription column: 'text'
Filtering audio > 30.0s + empty transcriptions...
  train: 79857 → 79641
  test:  1993 → 1962
  filtering done in 0.1s


### 3.2 Subset selection by hours

In [6]:
# === CELL 6: Subset selection helper ===
def select_hours(ds: Dataset, target_hours: float, seed: int = 42) -> Dataset:
    """Sample a subset of `ds` such that the total audio duration ≈ target_hours.

    Strategy: shuffle deterministically (seed), then accumulate samples until the
    cumulative duration reaches target_hours. Returns a Dataset.
    """
    target_seconds = target_hours * 3600
    shuffled = ds.shuffle(seed=seed)

    # First-pass cumulative duration computation
    # (Avoids materializing all audio arrays; uses dataset metadata when possible.)
    cum_sec = 0.0
    n_selected = 0
    for i, ex in enumerate(shuffled):
        audio = ex["audio"]
        dur = len(audio["array"]) / audio["sampling_rate"]
        cum_sec += dur
        n_selected = i + 1
        if cum_sec >= target_seconds:
            break

    selected = shuffled.select(range(n_selected))
    actual_hours = cum_sec / 3600
    return selected, actual_hours


print("Subset selector defined.")


Subset selector defined.


### 3.3 Build train / val / WER₀ / periodic-eval splits

This mirrors the Qwen2-Audio setup exactly so the splits are byte-identical (same seeds, same dataset):

| split | source | size | used for |
|---|---|---|---|
| `train_set` | publisher's train minus 2000 val samples, then sampled to ~`DATA_SCALE_HOURS` | ~7,899 @ 10h | training |
| `val_set` | publisher's train, 2000 samples carved with seed=42 | 2000 | source of WER₀ + periodic eval |
| `wer0_subset` | shuffled val_set, first `MARCO_EVAL_SUBSET_SIZE` | 200 @ 10h | initial-LR WER₀ |
| `periodic_eval_subset` | same as `wer0_subset` (FIXED across training) | 200 @ 10h | Marco-ASR callback |
| `test_set` | publisher's official test | 1,993 | final headline WER |


In [7]:
# === CELL 7: Build all splits ===
print(f"Carving val_set ({VAL_SET_SIZE} samples, seed={SEED})...")
splits = moulsot_filt["train"].train_test_split(test_size=VAL_SET_SIZE, seed=SEED)
full_train_set = splits["train"]
val_set = splits["test"]
print(f"  full_train: {len(full_train_set)}")
print(f"  val_set:    {len(val_set)}")

# Sample training set to ~DATA_SCALE_HOURS hours
print(f"Sampling training set to ~{DATA_SCALE_HOURS}h (this iterates audio durations, may take a few min)...")
t0 = time.time()
train_set, actual_train_hours = select_hours(full_train_set, DATA_SCALE_HOURS, seed=SEED)
print(f"  train_set: {len(train_set)} samples, ~{actual_train_hours:.2f}h (target {DATA_SCALE_HOURS}h)")
print(f"  selection took {time.time()-t0:.1f}s")

# WER₀ subset + periodic eval subset (SAME indices, fixed across the run)
print(f"Sampling {MARCO_EVAL_SUBSET_SIZE}-utt eval subset from val_set (seed={SEED})...")
shuffled_val = val_set.shuffle(seed=SEED)
wer0_subset = shuffled_val.select(range(min(MARCO_EVAL_SUBSET_SIZE, len(shuffled_val))))
periodic_eval_subset = wer0_subset  # same dataset; periodic eval is fixed across training
print(f"  wer0_subset / periodic_eval_subset: {len(wer0_subset)} samples")

# Test set (untouched until final post-training eval)
test_set = moulsot_filt["test"]
print(f"\nDataset summary:")
print(f"  train_set:             {len(train_set)} samples (~{actual_train_hours:.2f}h)")
print(f"  val_set (held out):    {len(val_set)} samples")
print(f"  wer0_subset:           {len(wer0_subset)} samples (Marco-ASR initial LR)")
print(f"  periodic_eval_subset:  {len(periodic_eval_subset)} samples (Marco-ASR every {MARCO_EVAL_STEPS} steps)")
print(f"  test_set:              {len(test_set)} samples (sealed until post-training eval)")


Carving val_set (2000 samples, seed=42)...
  full_train: 77641
  val_set:    2000
Sampling training set to ~10h (this iterates audio durations, may take a few min)...
  train_set: 7892 samples, ~10.00h (target 10h)
  selection took 59.4s
Sampling 200-utt eval subset from val_set (seed=42)...
  wer0_subset / periodic_eval_subset: 200 samples

Dataset summary:
  train_set:             7892 samples (~10.00h)
  val_set (held out):    2000 samples
  wer0_subset:           200 samples (Marco-ASR initial LR)
  periodic_eval_subset:  200 samples (Marco-ASR every 100 steps)
  test_set:              1962 samples (sealed until post-training eval)


### 3.4 Casablanca Morocco subset for OOD evaluation

In [8]:
# === CELL 8: Load Casablanca OOD eval slice ===
print(f"Loading {CASABLANCA_DATASET_ID} subset={CASABLANCA_SUBSET}...")
t0 = time.time()
casablanca_ood = None
try:
    casablanca = load_dataset(CASABLANCA_DATASET_ID, CASABLANCA_SUBSET, split="test")
    print(f"  loaded in {time.time()-t0:.1f}s, {len(casablanca)} samples")
    cas_text_col = _find_text_col(casablanca)
    if cas_text_col != "transcription":
        casablanca = casablanca.rename_columns({cas_text_col: "transcription"})
    casablanca = casablanca.cast_column("audio", Audio(sampling_rate=16000))
    casablanca = casablanca.filter(_is_valid, num_proc=4)
    print(f"  after filter: {len(casablanca)} samples")
    if len(casablanca) > CASABLANCA_OOD_MAX_SAMPLES:
        casablanca_ood = casablanca.shuffle(seed=SEED).select(range(CASABLANCA_OOD_MAX_SAMPLES))
    else:
        casablanca_ood = casablanca
    print(f"  casablanca_ood: {len(casablanca_ood)} samples (capped at {CASABLANCA_OOD_MAX_SAMPLES})")
except Exception as e:
    print(f"  failed to load Casablanca OOD set: {e}")
    print(f"  OOD evaluation will be skipped at post-training stage.")


Loading UBC-NLP/Casablanca subset=Morocco...
  loaded in 2.0s, 1045 samples
  after filter: 1045 samples
  casablanca_ood: 500 samples (capped at 500)


## 4. Evaluation utilities

In [9]:
# === CELL 9: Casablanca normalization ===
# IDENTICAL to qwen2audio_darija_finetune_v2.ipynb for byte-identical metric computation.
_DIACRITICS_RE = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]"
)
_HAMZA_MADDA_RE = re.compile(r"[إأآٱ]")
_PUNCT_RE = re.compile(r"[^\w\s%@]")
_WS_RE = re.compile(r"\s+")
_EASTERN_TO_WESTERN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")


def casablanca_normalize(text: str) -> str:
    """Casablanca paper normalization (Talafha et al., 2024, arXiv:2410.04527).

    Steps:
      1. Remove Arabic diacritics, hamzas, maddas (keep bare alif).
      2. Convert Eastern → Western Arabic numerals.
      3. Lowercase Latin characters (Darija mixes scripts).
      4. Remove all punctuation EXCEPT % and @ (semantically meaningful).
      5. Normalize whitespace.
    """
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = _DIACRITICS_RE.sub("", text)
    text = _HAMZA_MADDA_RE.sub("ا", text)
    text = text.translate(_EASTERN_TO_WESTERN)
    text = text.lower()
    text = _PUNCT_RE.sub(" ", text)
    text = _WS_RE.sub(" ", text).strip()
    return text


# Sanity tests (matched against qwen2audio notebook outputs)
test_inputs = [
    "السَّلامُ عَلَيْكُم",       # diacritics
    "أين كنتي؟",                  # hamza
    "Hello salam, كيف الحال?",    # mixed script + punctuation
    "في 2024 العام الماضي",       # Western digits (passthrough)
    "في ٢٠٢٤ العام الماضي",       # Eastern digits → Western
]
print("Normalization sanity check:")
for inp in test_inputs:
    print(f"  {inp:40s} → {casablanca_normalize(inp)!r}")


Normalization sanity check:
  السَّلامُ عَلَيْكُم                      → 'السلام عليكم'
  أين كنتي؟                                → 'اين كنتي'
  Hello salam, كيف الحال?                  → 'hello salam كيف الحال'
  في 2024 العام الماضي                     → 'في 2024 العام الماضي'
  في ٢٠٢٤ العام الماضي                     → 'في 2024 العام الماضي'


### 4.2 Code-switching detection

In [10]:
# === CELL 10: Code-switching detection ===
# IDENTICAL to qwen2audio_darija_finetune_v2.ipynb.
_ARABIC_RANGE_RE = re.compile(r"[\u0600-\u06FF]")
_LATIN_RANGE_RE = re.compile(r"[A-Za-z]")


def is_code_switched(raw_reference: str) -> bool:
    """True iff reference contains both Arabic AND Latin characters.

    Pure-Latin (e.g. fully French)   → False (monolingual)
    Pure-Arabic (Darija/MSA)         → False (monolingual)
    Mixed                            → True  (code-switched)

    Operates on RAW (pre-normalization) text so case + script are preserved.
    """
    if not raw_reference:
        return False
    has_arabic = bool(_ARABIC_RANGE_RE.search(raw_reference))
    has_latin = bool(_LATIN_RANGE_RE.search(raw_reference))
    return has_arabic and has_latin


# Sanity tests
assert not is_code_switched("بسم الله")
assert not is_code_switched("hello world")
assert is_code_switched("اشتريت l'voiture")
assert is_code_switched("Bonjour كيف الحال")
assert not is_code_switched("")
print("CS detection sanity tests passed ✓")


CS detection sanity tests passed ✓


### 4.3 Inference function (Whisper-specific batched generate)

In [11]:
# === CELL 11: Inference function (with timing) ===
@torch.no_grad()
def whisper_inference_batched(
    audios_with_sr: List[Tuple[np.ndarray, int]],
    model,
    processor,
    forced_decoder_ids,
    device: str = DEVICE,
    max_new_tokens: int = MAX_GEN_TOKENS,
    num_beams: int = 1,
) -> List[str]:
    """Batched Whisper inference. Returns list of decoded predictions.

    Why batched: at 200 utterances and ~0.3s/utt for greedy Whisper-LV3 on A40,
    sequential is ~60s per eval; batch=4 is ~15-20s. Marco-ASR calls this every
    eval, so the savings compound.

    Args:
        audios_with_sr: list of (audio_array, sampling_rate) tuples. All must be 16kHz.
        forced_decoder_ids: list of (position, token_id) tuples from
                            processor.get_decoder_prompt_ids(language=..., task=...).
                            This is what tells Whisper to transcribe in Arabic (for Darija).
        num_beams: 1 = greedy (fast, matches Marco-ASR callback). >1 = beam search.
    """
    # All inputs must share the same sample rate (we already cast to 16kHz)
    target_sr = processor.feature_extractor.sampling_rate
    arrays = []
    for audio_array, sr in audios_with_sr:
        a = audio_array.astype(np.float32, copy=False)
        if sr != target_sr:
            a = librosa.resample(a, orig_sr=sr, target_sr=target_sr)
        arrays.append(a)

    # Extract log-mel features (B, 128, 3000) padded to 30s
    input_features = processor.feature_extractor(
        arrays,
        sampling_rate=target_sr,
        return_tensors="pt",
    ).input_features

    # Move to device + match model dtype
    model_dtype = next(model.parameters()).dtype
    input_features = input_features.to(device, dtype=model_dtype)

    generated_ids = model.generate(
        input_features=input_features,
        forced_decoder_ids=forced_decoder_ids,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        do_sample=False,
    )
    preds = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    return preds


print("Inference function defined: whisper_inference_batched()")


Inference function defined: whisper_inference_batched()


### 4.4 Bootstrap confidence intervals

In [12]:
# === CELL 12: Bootstrap CI ===
# IDENTICAL to qwen2audio_darija_finetune_v2.ipynb — same pair-resampling, same percentile CI.
def bootstrap_metric(
    pairs: List[Tuple[str, str]],
    metric_fn: Callable[[List[str], List[str]], float],
    n_resamples: int = BOOTSTRAP_RESAMPLES,
    ci: float = BOOTSTRAP_CI,
    seed: int = SEED,
) -> Optional[Dict[str, float]]:
    """Bootstrap a corpus-level metric with paired resampling.

    pairs:      list of (normalized_ref, normalized_hyp) tuples
    metric_fn:  callable like jiwer_wer / jiwer_cer
    """
    if not pairs:
        return None
    rng = np.random.RandomState(seed)
    n = len(pairs)
    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]

    point = float(metric_fn(refs, hyps))

    boot_values = []
    for _ in range(n_resamples):
        idx = rng.choice(n, size=n, replace=True)
        b_refs = [refs[i] for i in idx]
        b_hyps = [hyps[i] for i in idx]
        try:
            boot_values.append(float(metric_fn(b_refs, b_hyps)))
        except Exception:
            continue

    alpha = 1 - ci
    lo = float(np.percentile(boot_values, 100 * alpha / 2))
    hi = float(np.percentile(boot_values, 100 * (1 - alpha / 2)))

    return {
        "point": point,
        "ci_low": lo,
        "ci_high": hi,
        "n": n,
        "n_resamples": n_resamples,
    }


print("bootstrap_metric defined.")


bootstrap_metric defined.


### 4.5 Evaluation function with code-switching partitioning

In [13]:
# === CELL 13: Evaluation with CS partitioning + bootstrap ===
def evaluate_with_partitioning(
    dataset,
    model,
    processor,
    *,
    forced_decoder_ids=None,
    transcription_field: str = "transcription",
    device: str = DEVICE,
    max_samples: Optional[int] = None,
    eval_batch_size: int = MARCO_EVAL_BATCH_SIZE,
    max_new_tokens: int = MAX_GEN_TOKENS,
    num_beams: int = 1,
    do_bootstrap: bool = True,
    verbose: bool = False,
    eval_name: str = "eval",
    save_details: bool = True,
    results_dir: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate Whisper model on dataset with code-switching partitioning.

    Mirrors the Qwen2-Audio evaluate_with_partitioning() in structure and outputs
    so paired-bootstrap comparisons across the two models are well-defined.

    Returns a dict with keys: full / code_switched / monolingual,
    each with {wer, cer, n, samples, raw_pairs}.

    forced_decoder_ids defaults to module-global FORCED_DECODER_IDS if None.
    """
    if forced_decoder_ids is None:
        forced_decoder_ids = FORCED_DECODER_IDS
    if results_dir is None:
        results_dir = RESULTS_DIR

    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))

    if verbose:
        print(f"[{eval_name}] running inference on {n} samples (batch={eval_batch_size}, beams={num_beams})...")

    # Gather audio + refs first (decouple loading from generation)
    audios_with_sr = []
    raw_refs = []
    for i in range(n):
        ex = dataset[i]
        audio = ex["audio"]["array"]
        if not isinstance(audio, np.ndarray):
            audio = np.asarray(audio)
        sr = ex["audio"]["sampling_rate"]
        ref = ex.get(transcription_field, "") or ""
        audios_with_sr.append((audio, sr))
        raw_refs.append(ref)

    # Batched generation
    all_preds = []
    inference_starts = []
    inference_ends = []
    durations = []
    for batch_start in range(0, len(audios_with_sr), eval_batch_size):
        batch = audios_with_sr[batch_start:batch_start + eval_batch_size]
        t0 = time.perf_counter()
        try:
            preds = whisper_inference_batched(
                batch, model=model, processor=processor,
                forced_decoder_ids=forced_decoder_ids,
                device=device, max_new_tokens=max_new_tokens, num_beams=num_beams,
            )
        except Exception as e:
            print(f"  [batch {batch_start} generate failed]: {type(e).__name__}: {e}")
            preds = [""] * len(batch)
        t1 = time.perf_counter()
        per_sample = (t1 - t0) / len(batch)
        for (audio, sr) in batch:
            inference_starts.append(t0)
            inference_ends.append(t1)
            durations.append(len(audio) / sr)
        all_preds.extend(preds)

    # Normalize + partition + collect per-utterance details
    details = []
    full_pairs, cs_pairs, mono_pairs = [], [], []
    for raw_ref, raw_pred, dur in zip(raw_refs, all_preds, durations):
        norm_ref = casablanca_normalize(raw_ref)
        norm_pred = casablanca_normalize(raw_pred)
        is_cs = is_code_switched(raw_ref)
        details.append({
            "raw_ref": raw_ref,
            "raw_pred": raw_pred,
            "norm_ref": norm_ref,
            "norm_pred": norm_pred,
            "is_cs": is_cs,
            "audio_dur_sec": dur,
        })
        if not norm_ref:
            continue
        pair = (norm_ref, norm_pred)
        full_pairs.append(pair)
        if is_cs:
            cs_pairs.append(pair)
        else:
            mono_pairs.append(pair)

    def _partition_metrics(pairs, label):
        if not pairs:
            return {"wer": None, "cer": None, "n": 0}
        if do_bootstrap:
            return {
                "wer": bootstrap_metric(pairs, jiwer_wer),
                "cer": bootstrap_metric(pairs, jiwer_cer),
                "n": len(pairs),
            }
        else:
            refs = [p[0] for p in pairs]
            hyps = [p[1] for p in pairs]
            return {
                "wer": {"point": float(jiwer_wer(refs, hyps)), "n": len(pairs)},
                "cer": {"point": float(jiwer_cer(refs, hyps)), "n": len(pairs)},
                "n": len(pairs),
            }

    metrics = {
        "full":          _partition_metrics(full_pairs, "full"),
        "code_switched": _partition_metrics(cs_pairs, "code_switched"),
        "monolingual":   _partition_metrics(mono_pairs, "monolingual"),
    }

    # Timing
    total_audio_sec = sum(durations) if durations else 0.0
    total_inference_sec = (inference_ends[-1] - inference_starts[0]) if inference_ends else 0.0
    mean_per_utt = (total_inference_sec / n) if n > 0 else 0.0
    rtf = (total_inference_sec / total_audio_sec) if total_audio_sec > 0 else None

    summary = {
        "eval_name": eval_name,
        "n_samples": n,
        "n_valid": len(full_pairs),
        "n_cs": len(cs_pairs),
        "n_mono": len(mono_pairs),
        "language": LANGUAGE,
        "task": TASK,
        "num_beams": num_beams,
        "total_inference_sec": total_inference_sec,
        "mean_inference_sec_per_utt": mean_per_utt,
        "rtf": rtf,
        "metrics": metrics,
    }

    if save_details:
        os.makedirs(results_dir, exist_ok=True)
        with open(os.path.join(results_dir, f"{eval_name}_details.json"), "w", encoding="utf-8") as f:
            json.dump(details, f, indent=2, ensure_ascii=False)
        with open(os.path.join(results_dir, f"{eval_name}_summary.json"), "w") as f:
            json.dump(summary, f, indent=2)
        if verbose:
            print(f"  saved {eval_name}_details.json + _summary.json")

    if verbose:
        print(f"\n[{eval_name}] results:")
        for partition in ["full", "code_switched", "monolingual"]:
            m = metrics[partition]
            if m["wer"] is None:
                print(f"  {partition:15s}: n=0")
                continue
            w = m["wer"]
            c = m["cer"]
            if "ci_low" in w:
                print(f"  {partition:15s} (n={m['n']:5d}): "
                      f"WER {w['point']*100:6.2f}% [{w['ci_low']*100:6.2f}, {w['ci_high']*100:6.2f}]  "
                      f"CER {c['point']*100:6.2f}% [{c['ci_low']*100:6.2f}, {c['ci_high']*100:6.2f}]")
            else:
                print(f"  {partition:15s} (n={m['n']:5d}): "
                      f"WER {w['point']*100:6.2f}%  CER {c['point']*100:6.2f}%")
        if rtf is not None:
            print(f"  RTF: {rtf:.3f} (inference / audio)")

    return summary


print("evaluate_with_partitioning defined.")


evaluate_with_partitioning defined.


## 5. Zero-shot WER₀ on val subset

Whisper-LV3 zero-shot on the MoulSot **test set** has already been computed in the standalone notebook. Here we compute WER₀ on the **`wer0_subset`** (a 200-utterance slice of the held-out val set) — this is what seeds Marco-ASR Algorithm 2's initial LRs.

Key methodological point: this is the **same dataset** the Marco-ASR callback will evaluate on every 100 steps, so WER₀ is directly comparable to the periodic-eval WERs that drive the adaptive LR schedule. The test set is **not touched** here.


In [14]:
# === CELL 14: Zero-shot WER₀ on val subset ===
if MODE in ("fine_tune", "wer0_only"):
    print(f">>> Computing WER₀ on {len(wer0_subset)} utterances of val_set (Whisper greedy, language={LANGUAGE!r}) <<<")
    wer0_results = evaluate_with_partitioning(
        wer0_subset,
        model=model,
        processor=processor,
        forced_decoder_ids=FORCED_DECODER_IDS,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        max_new_tokens=MAX_GEN_TOKENS,
        num_beams=1,
        do_bootstrap=False,        # not needed for a single seed point
        verbose=True,
        eval_name=f"wer0_whisper_{DATA_SCALE_HOURS}h",
        save_details=True,
    )
    wer0 = wer0_results["metrics"]["full"]["wer"]["point"]
    cer0 = wer0_results["metrics"]["full"]["cer"]["point"]
    print(f"\n  WER₀ = {wer0*100:.2f}%")
    print(f"  CER₀ = {cer0*100:.2f}%")
else:
    print(f"MODE={MODE}: skipping WER₀ computation.")
    wer0_results = None
    wer0 = None
    cer0 = None


>>> Computing WER₀ on 200 utterances of val_set (Whisper greedy, language='arabic') <<<
[wer0_whisper_10h] running inference on 200 samples (batch=4, beams=1)...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  saved wer0_whisper_10h_details.json + _summary.json

[wer0_whisper_10h] results:
  full            (n=  200): WER  77.14%  CER  44.34%
  code_switched   (n=   48): WER  82.96%  CER  56.01%
  monolingual     (n=  152): WER  74.36%  CER  38.73%
  RTF: 0.114 (inference / audio)

  WER₀ = 77.14%
  CER₀ = 44.34%


## 6. Marco-ASR Algorithm 2 initialization

Algorithm 2 (Ni et al. 2025) for encoder-decoder ASR computes **two** initial LRs from WER₀:

$$
\eta_{\text{enc}} = \eta_{\text{enc,min}} + (\eta_{\text{enc,max}} - \eta_{\text{enc,min}}) \cdot \text{clip}(\text{WER}, 0, 1)
$$
$$
\eta_{\text{dec}} = \eta_{\text{dec,min}} + (\eta_{\text{dec,max}} - \eta_{\text{dec,min}}) \cdot \text{clip}(\text{WER}, 0, 1)
$$

The intuition: when error is high, both LRs are at the top of their ranges; as the model improves, both LRs anneal proportionally. The two ranges are configured independently because Whisper's encoder is already a strong audio representation extractor (small LR suffices to refine) while the decoder must learn the Darija output distribution (needs a larger LR).


In [15]:
# === CELL 15: Compute initial LRs from WER₀ (Algorithm 2) ===
def marco_asr_lr_alg2(
    wer_fraction: float,
    lr_enc_min: float = MARCO_LR_ENC_MIN,
    lr_enc_max: float = MARCO_LR_ENC_MAX,
    lr_dec_min: float = MARCO_LR_DEC_MIN,
    lr_dec_max: float = MARCO_LR_DEC_MAX,
) -> Tuple[float, float]:
    """Marco-ASR Algorithm 2: independent encoder + decoder LRs from WER."""
    wer_clipped = max(0.0, min(1.0, wer_fraction))
    lr_enc = lr_enc_min + (lr_enc_max - lr_enc_min) * wer_clipped
    lr_dec = lr_dec_min + (lr_dec_max - lr_dec_min) * wer_clipped
    return lr_enc, lr_dec


initial_lr_enc = MARCO_LR_ENC_MAX
initial_lr_dec = MARCO_LR_DEC_MAX

if MODE in ("fine_tune", "wer0_only") and wer0 is not None:
    initial_lr_enc, initial_lr_dec = marco_asr_lr_alg2(wer0)
    print(f"\nMarco-ASR Algorithm 2 initialization (WER₀ = {wer0*100:.2f}%):")
    print(f"  Initial LR (encoder): {initial_lr_enc:.3e}  (range [{MARCO_LR_ENC_MIN:.1e}, {MARCO_LR_ENC_MAX:.1e}])")
    print(f"  Initial LR (decoder): {initial_lr_dec:.3e}  (range [{MARCO_LR_DEC_MIN:.1e}, {MARCO_LR_DEC_MAX:.1e}])")
    print(f"  Ratio dec/enc:        {initial_lr_dec/initial_lr_enc:.1f}× (decoder LR ~10× encoder LR by design)")
else:
    print(f"MODE={MODE}: using LR maxes as initial values (WER₀ not computed)")
    print(f"  Initial LR (encoder): {initial_lr_enc:.3e}")
    print(f"  Initial LR (decoder): {initial_lr_dec:.3e}")



Marco-ASR Algorithm 2 initialization (WER₀ = 77.14%):
  Initial LR (encoder): 7.737e-06  (range [1.0e-07, 1.0e-05])
  Initial LR (decoder): 7.737e-05  (range [1.0e-06, 1.0e-04])
  Ratio dec/enc:        10.0× (decoder LR ~10× encoder LR by design)


## 7. Training preparation

### 7.1 Whisper data collator

Whisper-specific differences from a generic Seq2Seq collator:

1. **Audio**: `feature_extractor` produces log-mel spectrograms of shape `(batch, 128, 3000)` (Whisper-LV3 has 128 mel bins, fixed 30s window).
2. **Labels**: tokenized with the prefix tokens already set on the tokenizer — labels look like `[<|sot|>, <|ar|>, <|transcribe|>, <|notimestamps|>, …text…, <|eot|>]`.
3. **Strip leading `<|sot|>`**: `shift_tokens_right` inside the model's forward pass *prepends* the decoder_start_token (`<|sot|>`). If labels already have it, we'd get a duplicate. The standard recipe (Sanchit Gandhi's blog, HF docs) is to drop the leading `<|sot|>` from labels here.
4. **Padding mask**: replace pad-token positions with `-100` so they're ignored by cross-entropy.


In [16]:
# === CELL 16: Whisper data collator ===
@dataclass
class WhisperDataCollator:
    """Whisper-specific data collator. Produces input_features + labels.

    Standard recipe (Sanchit Gandhi 2022, HF docs):
      1. feature_extractor(audio_arrays)  →  (B, 128, 3000) log-mel
      2. tokenizer(transcriptions)        →  variable-length ids with prefix tokens
      3. pad labels to longest in batch, mask padding with -100 (ignored in CE loss)
      4. drop leading <|sot|>: shift_tokens_right inside forward pass re-adds it
    """
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        # --- Audio features ---
        audio_arrays = [f["audio"]["array"] for f in features]
        target_sr = self.processor.feature_extractor.sampling_rate
        sampling_rates = [f["audio"]["sampling_rate"] for f in features]
        # Resample any off-rate samples to target_sr (shouldn't happen since we cast to 16kHz, but safe)
        for i, (a, sr) in enumerate(zip(audio_arrays, sampling_rates)):
            if sr != target_sr:
                audio_arrays[i] = librosa.resample(
                    np.asarray(a, dtype=np.float32), orig_sr=sr, target_sr=target_sr
                )
        input_features = self.processor.feature_extractor(
            audio_arrays,
            sampling_rate=target_sr,
            return_tensors="pt",
        ).input_features

        # --- Labels: tokenize transcriptions ---
        texts = [f["transcription"] for f in features]
        tokenized = self.processor.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=448,             # Whisper max_target_positions
        )
        labels = tokenized.input_ids
        attention_mask = tokenized.attention_mask

        # Mask padding tokens with -100 for cross-entropy
        labels = labels.masked_fill(attention_mask == 0, -100)

        # Drop leading <|sot|> — shift_tokens_right will re-add it in forward pass
        # (Avoids a duplicate SOT in the decoder input.)
        if (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]

        return {
            "input_features": input_features,
            "labels": labels,
        }


if MODE == "fine_tune":
    data_collator = WhisperDataCollator(
        processor=processor,
        decoder_start_token_id=DECODER_START_TOKEN_ID,
    )

    # Sanity check
    print("Data collator sanity check (1 sample)...")
    sample_batch = data_collator([train_set[0]])
    print(f"  input_features:  {tuple(sample_batch['input_features'].shape)}  {sample_batch['input_features'].dtype}")
    print(f"  labels:          {tuple(sample_batch['labels'].shape)}  {sample_batch['labels'].dtype}")
    decoded = processor.tokenizer.decode([t for t in sample_batch["labels"][0].tolist() if t != -100])
    print(f"  labels decoded:  {decoded!r}")
    print(f"  raw transcript:  {train_set[0]['transcription']!r}")
    print("✓ Collator sanity check passed")

    # Larger batch test (catches padding shape mismatches)
    print("\nData collator batch test (4 samples)...")
    sample_batch4 = data_collator([train_set[i] for i in range(4)])
    print(f"  input_features:  {tuple(sample_batch4['input_features'].shape)}  (expected: (4, 128, 3000))")
    print(f"  labels:          {tuple(sample_batch4['labels'].shape)}  (varies by max length)")
    assert sample_batch4['input_features'].shape == (4, 128, 3000), "input_features shape mismatch"
    print("✓ Batch sanity check passed")


Data collator sanity check (1 sample)...
  input_features:  (1, 128, 3000)  torch.float32
  labels:          (1, 18)  torch.int64
  labels decoded:  '<|ar|><|transcribe|><|notimestamps|>ف distributed system وكيتعتبر like the most common واحد<|endoftext|>'
  raw transcript:  'ف distributed system وكيتعتبر like the most common واحد'
✓ Collator sanity check passed

Data collator batch test (4 samples)...
  input_features:  (4, 128, 3000)  (expected: (4, 128, 3000))
  labels:          (4, 117)  (varies by max length)
✓ Batch sanity check passed


### 7.2 Apply LoRA

LoRA config is **identical** to the Qwen2-Audio run for direct comparability (r=32, alpha=32, dropout=0.1, rsLoRA=True, `target_modules="all-linear"`). The only differences are:

- `task_type=SEQ_2_SEQ_LM` instead of `CAUSAL_LM` (Whisper is encoder-decoder)
- The audit cell partitions LoRA modules into encoder / decoder / other for sanity checking


In [17]:
# === CELL 17: Apply LoRA ===
if MODE == "fine_tune":
    print(f"Applying LoRA (r={LORA_RANK}, alpha={LORA_ALPHA}, rsLoRA={USE_RSLORA}) to all linear layers...")

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules="all-linear",
        lora_dropout=LORA_DROPOUT,
        bias=LORA_BIAS,
        #task_type=TaskType.SEQ_2_SEQ_LM,        # ↔ Qwen2-Audio uses CAUSAL_LM
        use_rslora=USE_RSLORA,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Audit: how many LoRA modules ended up on encoder vs decoder?
    audit = {"encoder": [], "decoder": [], "other": []}
    for name, module in model.named_modules():
        if "lora_A.default" not in name:
            continue
        base = name.replace(".lora_A.default", "")
        if ".encoder." in base:
            audit["encoder"].append(base)
        elif ".decoder." in base:
            audit["decoder"].append(base)
        else:
            audit["other"].append(base)

    print(f"\nLoRA module audit:")
    print(f"  encoder LoRA modules: {len(audit['encoder'])}")
    print(f"  decoder LoRA modules: {len(audit['decoder'])}")
    print(f"  other LoRA modules:   {len(audit['other'])}")
    print(f"  total:                {sum(len(v) for v in audit.values())}")

    # Save audit JSON
    audit_path = os.path.join(RESULTS_DIR, f"lora_targets_whisper_{DATA_SCALE_HOURS}h.json")
    with open(audit_path, "w") as f:
        json.dump({k: len(v) for k, v in audit.items()} | {"modules": audit}, f, indent=2)
    print(f"  audit saved to {audit_path}")


Applying LoRA (r=32, alpha=32, rsLoRA=True) to all linear layers...
trainable params: 57,671,680 || all params: 1,601,162,240 || trainable%: 3.6019

LoRA module audit:
  encoder LoRA modules: 192
  decoder LoRA modules: 320
  other LoRA modules:   0
  total:                512
  audit saved to /workspace/outputs/results/lora_targets_whisper_10h.json


### 7.3 Custom optimizer with named param groups

This cell is **new** vs the Qwen2-Audio notebook. Marco-ASR Algorithm 2 requires setting **independent LRs** on encoder vs decoder parameters. HF Trainer's default optimizer puts everything in one param group, so we construct AdamW manually with two named groups:

- Group `"encoder"`: parameters whose name contains `.encoder.`
- Group `"decoder"`: everything else (decoder layers, embeddings, proj_out)

The `"name"` field is preserved by PyTorch's `Optimizer.load_state_dict`, so the named groups survive checkpoint save/resume.

We pass the pre-built optimizer to `Trainer(optimizers=(opt, None))` — Trainer's `create_optimizer` then skips its own optimizer creation, and the LR scheduler is auto-built from our optimizer (with both base_lrs preserved).


In [18]:
# === CELL 17b: Custom optimizer with encoder/decoder param groups ===
def create_whisper_optimizer_with_groups(
    model,
    lr_enc_init: float,
    lr_dec_init: float,
    weight_decay: float = WEIGHT_DECAY,
) -> torch.optim.Optimizer:
    """AdamW with two named param groups: 'encoder' and 'decoder'.

    Parameter classification:
      - 'encoder' group: name contains '.encoder.' (e.g. 'base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight')
      - 'decoder' group: everything else (decoder layers, embeddings, proj_out, etc.)

    PyTorch preserves arbitrary extra fields in param_groups (including 'name')
    through state_dict save/load, so this structure survives checkpoint resume.
    """
    encoder_params = []
    decoder_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if ".encoder." in name:
            encoder_params.append(p)
        else:
            decoder_params.append(p)

    if not encoder_params:
        raise RuntimeError(
            "No encoder LoRA parameters found. Check that LoRA was applied to encoder layers."
        )
    if not decoder_params:
        raise RuntimeError(
            "No decoder LoRA parameters found. Check that LoRA was applied to decoder layers."
        )

    print(f"  Encoder param tensors: {len(encoder_params):4d}  ({sum(p.numel() for p in encoder_params)/1e6:.2f}M params)")
    print(f"  Decoder param tensors: {len(decoder_params):4d}  ({sum(p.numel() for p in decoder_params)/1e6:.2f}M params)")

    optimizer = torch.optim.AdamW([
        {
            "params": encoder_params,
            "lr": lr_enc_init,
            "weight_decay": weight_decay,
            "name": "encoder",
        },
        {
            "params": decoder_params,
            "lr": lr_dec_init,
            "weight_decay": weight_decay,
            "name": "decoder",
        },
    ])
    return optimizer


print("create_whisper_optimizer_with_groups defined.")


create_whisper_optimizer_with_groups defined.


### 7.4 Marco-ASR Algorithm 2 callback

Mirrors `MarcoASRCallback` from the Qwen2-Audio notebook in **all infrastructure**:

- Resume from log JSON (replay history forward, reconstruct `best_wer` and `no_improve_count`)
- Save best-WER checkpoint via `model.save_pretrained` on improvement
- Same patience / min_delta logic for early stopping
- Same `history` JSON format (with two extra fields: `lr_encoder`, `lr_decoder`)

The **only differences** from the Qwen2-Audio callback:

1. `_compute_lrs()` returns a **pair** `(lr_enc, lr_dec)` instead of a scalar
2. `_set_lrs()` walks `optimizer.param_groups` by `name` field and updates each group + the corresponding `scheduler.base_lrs[i]`
3. `_evaluate()` calls `whisper_inference_batched` (Whisper-specific generate with forced_decoder_ids) instead of `run_inference_batched` (Qwen2-Audio chat-template generate)
4. Log entries carry `lr_encoder` and `lr_decoder` separately for the plotting cell


In [19]:
# === CELL 18: Marco-ASR Algorithm 2 callback ===
class MarcoASRCallbackForWhisper(TrainerCallback):
    """Marco-ASR Algorithm 2 (Ni et al. 2025) for encoder-decoder ASR.

    Periodically evaluates the model on a fixed val subset, recomputes
    encoder + decoder LRs from current WER (independent ranges), and applies
    early stopping based on best-so-far WER + patience.

    Identical infrastructure to MarcoASRCallback for Qwen2-Audio:
      - Resume from log JSON (no_improve replay)
      - Best-checkpoint save on improvement
      - Same patience / min_delta logic
    Differences:
      - Two LR ranges (encoder, decoder) instead of one
      - Updates optimizer.param_groups by 'name' field
      - Uses Whisper-specific generate (forced_decoder_ids)
    """

    def __init__(
        self,
        eval_dataset,
        processor,
        eval_steps: int = MARCO_EVAL_STEPS,
        eval_batch_size: int = MARCO_EVAL_BATCH_SIZE,
        lr_enc_min: float = MARCO_LR_ENC_MIN,
        lr_enc_max: float = MARCO_LR_ENC_MAX,
        lr_dec_min: float = MARCO_LR_DEC_MIN,
        lr_dec_max: float = MARCO_LR_DEC_MAX,
        patience: int = MARCO_PATIENCE,
        min_delta_pct: float = MARCO_MIN_DELTA_PCT,
        best_checkpoint_dir: Optional[str] = None,
        log_path: Optional[str] = None,
        language: str = LANGUAGE,
        task: str = TASK,
        max_new_tokens: int = MAX_GEN_TOKENS,
        resume_from_log: bool = False,
        verbose: bool = True,
    ):
        super().__init__()
        self.eval_dataset = eval_dataset
        self.processor = processor
        self.eval_steps = eval_steps
        self.eval_batch_size = eval_batch_size
        self.lr_enc_min = lr_enc_min
        self.lr_enc_max = lr_enc_max
        self.lr_dec_min = lr_dec_min
        self.lr_dec_max = lr_dec_max
        self.patience = patience
        self.min_delta_pct = min_delta_pct
        self.best_checkpoint_dir = best_checkpoint_dir
        self.log_path = log_path
        self.language = language
        self.task = task
        self.max_new_tokens = max_new_tokens
        self.resume_from_log = resume_from_log
        self.verbose = verbose

        # Pre-compute forced decoder IDs once
        self.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language, task=task)

        self.best_wer = float("inf")
        self.best_step = -1
        self.no_improve_count = 0
        self.history: List[Dict[str, Any]] = []
        self._train_start_time = None

    def _load_resume_state(self):
        """Load history JSON and replay forward to reconstruct best_wer + no_improve_count."""
        if not self.log_path or not os.path.exists(self.log_path):
            return
        try:
            with open(self.log_path, "r") as f:
                history = json.load(f)
        except Exception as e:
            if self.verbose:
                print(f"[MARCO-ASR] could not load resume log: {e}")
            return
        if not history:
            return
        best_wer = float("inf")
        best_step = -1
        no_improve = 0
        for record in history:
            wer_pct = record["wer"] * 100
            if wer_pct < (best_wer * 100 - self.min_delta_pct):
                best_wer = record["wer"]
                best_step = record["step"]
                no_improve = 0
            else:
                no_improve += 1
        self.history = history
        self.best_wer = best_wer
        self.best_step = best_step
        self.no_improve_count = no_improve
        if self.verbose:
            print(f"[MARCO-ASR] resumed: {len(history)} prior evals, "
                  f"best WER {best_wer*100:.2f}% at step {best_step}, "
                  f"no_improve_count={no_improve}/{self.patience}")

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_start_time = time.time()
        if self.resume_from_log:
            self._load_resume_state()
        if self.verbose:
            print(f"[MARCO-ASR] training begins; eval every {self.eval_steps} steps "
                  f"on {len(self.eval_dataset)} utt (batch={self.eval_batch_size})")
            print(f"[MARCO-ASR] encoder LR range: [{self.lr_enc_min:.2e}, {self.lr_enc_max:.2e}]")
            print(f"[MARCO-ASR] decoder LR range: [{self.lr_dec_min:.2e}, {self.lr_dec_max:.2e}]")
        return control

    @torch.no_grad()
    def _evaluate_wer(self, model) -> Dict[str, float]:
        """Run BATCHED inference on periodic eval subset → corpus-level WER/CER."""
        was_training = model.training
        model.eval()

        # Gather audios + refs
        all_audios_with_sr = []
        all_refs = []
        for ex in self.eval_dataset:
            try:
                audio = ex["audio"]["array"]
                if not isinstance(audio, np.ndarray):
                    audio = np.asarray(audio)
                sr = ex["audio"]["sampling_rate"]
                ref = ex.get("transcription", "") or ""
                all_audios_with_sr.append((audio, sr))
                all_refs.append(ref)
            except Exception as e:
                if self.verbose:
                    print(f"[MARCO-ASR] eval load error: {e}")
                continue

        # Batched generation
        all_preds = []
        for batch_start in range(0, len(all_audios_with_sr), self.eval_batch_size):
            batch = all_audios_with_sr[batch_start:batch_start + self.eval_batch_size]
            try:
                preds = whisper_inference_batched(
                    batch, model=model, processor=self.processor,
                    forced_decoder_ids=self.forced_decoder_ids,
                    max_new_tokens=self.max_new_tokens, num_beams=1,
                )
                all_preds.extend(preds)
            except Exception as e:
                if self.verbose:
                    print(f"[MARCO-ASR] eval generate error at batch {batch_start}: {e}")
                all_preds.extend([""] * len(batch))

        # Normalize + compute corpus-level metrics
        refs, hyps = [], []
        for ref, pred in zip(all_refs, all_preds):
            ref_n = casablanca_normalize(ref)
            hyp_n = casablanca_normalize(pred)
            if ref_n:
                refs.append(ref_n)
                hyps.append(hyp_n)

        if was_training:
            model.train()
        if not refs:
            return {"wer": float("nan"), "cer": float("nan"), "n": 0}
        return {
            "wer": float(jiwer_wer(refs, hyps)),
            "cer": float(jiwer_cer(refs, hyps)),
            "n": len(refs),
        }

    def _compute_lrs(self, wer_fraction: float) -> Tuple[float, float]:
        """Algorithm 2: independent encoder + decoder LRs from WER."""
        wer_clipped = max(0.0, min(1.0, wer_fraction))
        lr_enc = self.lr_enc_min + (self.lr_enc_max - self.lr_enc_min) * wer_clipped
        lr_dec = self.lr_dec_min + (self.lr_dec_max - self.lr_dec_min) * wer_clipped
        return lr_enc, lr_dec

    def _set_lrs(self, optimizer, lr_scheduler, lr_enc: float, lr_dec: float):
        """Update optimizer.param_groups (by 'name' field) AND lr_scheduler.base_lrs.

        With lr_scheduler_type='constant', the scheduler rewrites optimizer.lr at
        each step from its base_lrs. We must update BOTH for the LR change to stick.
        """
        for pg in optimizer.param_groups:
            if pg.get("name") == "encoder":
                pg["lr"] = lr_enc
            elif pg.get("name") == "decoder":
                pg["lr"] = lr_dec
        if lr_scheduler is not None and hasattr(lr_scheduler, "base_lrs"):
            # base_lrs is in the same order as optimizer.param_groups
            new_base_lrs = []
            for pg in optimizer.param_groups:
                if pg.get("name") == "encoder":
                    new_base_lrs.append(lr_enc)
                elif pg.get("name") == "decoder":
                    new_base_lrs.append(lr_dec)
                else:
                    new_base_lrs.append(pg["lr"])
            lr_scheduler.base_lrs = new_base_lrs

    def _save_log(self):
        if not self.log_path:
            return
        try:
            os.makedirs(os.path.dirname(self.log_path), exist_ok=True)
            with open(self.log_path, "w") as f:
                json.dump(self.history, f, indent=2)
        except Exception as e:
            if self.verbose:
                print(f"[MARCO-ASR] could not save log: {e}")

    def on_step_end(self, args, state, control, model=None, optimizer=None, lr_scheduler=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.eval_steps != 0:
            return control

        # Evaluate
        eval_t0 = time.perf_counter()
        m = self._evaluate_wer(model)
        eval_dt = time.perf_counter() - eval_t0
        wer = m["wer"]
        cer = m["cer"]

        # Compute + apply new LRs
        new_lr_enc, new_lr_dec = self._compute_lrs(wer)
        self._set_lrs(optimizer, lr_scheduler, new_lr_enc, new_lr_dec)

        # Improvement check
        wer_pct = wer * 100
        best_pct = self.best_wer * 100
        improved = wer_pct < best_pct - self.min_delta_pct

        # Log
        elapsed = time.time() - self._train_start_time
        self.history.append({
            "step": int(state.global_step),
            "wer": float(wer),
            "cer": float(cer),
            "n": m["n"],
            "lr_encoder": float(new_lr_enc),
            "lr_decoder": float(new_lr_dec),
            "elapsed_sec": float(elapsed),
            "improved": bool(improved),
        })
        self._save_log()

        # State + best-checkpoint save
        if improved:
            self.best_wer = wer
            self.best_step = int(state.global_step)
            self.no_improve_count = 0
            if self.best_checkpoint_dir:
                try:
                    model.save_pretrained(self.best_checkpoint_dir)
                except Exception as e:
                    if self.verbose:
                        print(f"[MARCO-ASR] could not save best checkpoint: {e}")
        else:
            self.no_improve_count += 1
            if self.no_improve_count >= self.patience:
                if self.verbose:
                    print(f"[MARCO-ASR] early stopping at step {state.global_step}: "
                          f"no improvement (>={self.min_delta_pct}pp) for "
                          f"{self.patience} consecutive evals")
                control.should_training_stop = True

        if self.verbose:
            arrow = "↓" if improved else "→"
            print(f"[MARCO-ASR step={state.global_step:5d}] {arrow} "
                  f"WER={wer_pct:6.2f}%  CER={cer*100:6.2f}%  "
                  f"LR_enc={new_lr_enc:.2e}  LR_dec={new_lr_dec:.2e}  "
                  f"best={self.best_wer*100:6.2f}% (step {self.best_step})  "
                  f"no_improve={self.no_improve_count}/{self.patience}  "
                  f"eval={eval_dt:.1f}s")
        return control

    def on_train_end(self, args, state, control, **kwargs):
        if self.verbose:
            print(f"\n[MARCO-ASR] training ended. Best WER: {self.best_wer*100:.2f}% at step {self.best_step}")
        return control


class DiskSpaceGuardCallback(TrainerCallback):
    """Abort training cleanly BEFORE a save would fail due to disk fill."""

    def __init__(self, output_dir, min_free_gb=8.0, verbose=True):
        self.output_dir = output_dir
        self.min_free_gb = min_free_gb
        self.verbose = verbose

    def _check_disk(self, label="save"):
        free_gb = shutil.disk_usage(self.output_dir).free / 1e9
        if self.verbose:
            print(f"[DISK-GUARD] before {label}: {free_gb:.1f} GB free")
        return free_gb

    def on_save(self, args, state, control, **kwargs):
        free_gb = self._check_disk(f"save at step {state.global_step}")
        if free_gb < self.min_free_gb:
            print(f"[DISK-GUARD] ABORTING: only {free_gb:.1f} GB free, need ≥ {self.min_free_gb} GB")
            control.should_training_stop = True
        return control

    def on_train_begin(self, args, state, control, **kwargs):
        self._check_disk("training start")
        return control


print("Callbacks defined: MarcoASRCallbackForWhisper, DiskSpaceGuardCallback")


Callbacks defined: MarcoASRCallbackForWhisper, DiskSpaceGuardCallback


### 7.5 Training arguments and Trainer

Key choices and how they differ from Qwen2-Audio:

- `lr_scheduler_type="constant"` — Marco-ASR overrides via the callback. Same as Qwen2-Audio.
- `gradient_checkpointing=False` — Whisper-LV3 at batch=4 fits on A40 (~15 GB activations), no need to trade speed for memory. Qwen2-Audio used it because 8.5B params + activations were tighter on memory.
- `save_steps=200` — matches the Qwen2-Audio v2 settings (post 10h run debugging). Earlier-frequent saves give better fallback granularity.
- `save_total_limit=3` — keep latest 3 trainer checkpoints (~1-1.5 GB total for Whisper LoRA, vs ~7-8 GB for Qwen2-Audio LoRA).
- `predict_with_generate=False` — Marco-ASR callback handles all eval. We don't use Seq2SeqTrainer because we don't need its built-in generation-based eval.
- `optimizers=(optimizer, None)` — we pre-build the optimizer with named param groups (encoder/decoder); Trainer's auto-built scheduler picks up both base_lrs from it.


In [20]:
# === CELL 19: Training arguments + Trainer ===
if MODE == "fine_tune":
    # Build the custom optimizer first (with named encoder/decoder param groups)
    print("Building custom optimizer with named param groups...")
    optimizer = create_whisper_optimizer_with_groups(
        model=model,
        lr_enc_init=initial_lr_enc,
        lr_dec_init=initial_lr_dec,
        weight_decay=WEIGHT_DECAY,
    )

    run_name = (
        f"whisper-lv3-darija-marco-{DATA_SCALE_HOURS}h-r{LORA_RANK}-"
        f"lrEnc{initial_lr_enc:.1e}-lrDec{initial_lr_dec:.1e}"
    )

    training_args = TrainingArguments(
        output_dir=TRAINER_OUTPUT_DIR,
        run_name=run_name,
        # --- Data / batch ---
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        # --- Optimizer ---
        # learning_rate here is a placeholder; the actual LRs come from
        # the param groups in our pre-built optimizer.
        learning_rate=initial_lr_dec,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="constant",            # Marco-ASR overrides via callback
        # --- Precision ---
        bf16=True,
        fp16=False,
        gradient_checkpointing=False,            # Whisper-LV3 LoRA fits comfortably without it
        # --- Eval / saving ---
        eval_strategy="no",                      # Marco-ASR runs its own eval
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        save_safetensors=True,
        # --- Logging ---
        logging_strategy="steps",
        logging_steps=25,
        logging_dir=os.path.join(LOGS_DIR, run_name),
        report_to=[],
        # --- Misc ---
        remove_unused_columns=False,             # the data collator handles column selection
        label_names=["labels"],
        dataloader_num_workers=0,
        seed=SEED,
        # --- Hub ---
        hub_model_id=HUB_MODEL_ID,
        push_to_hub=(HUB_MODEL_ID is not None),
    )

    # Marco-ASR Algorithm 2 callback
    marco_callback = MarcoASRCallbackForWhisper(
        eval_dataset=periodic_eval_subset,
        processor=processor,
        eval_steps=MARCO_EVAL_STEPS,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        lr_enc_min=MARCO_LR_ENC_MIN, lr_enc_max=MARCO_LR_ENC_MAX,
        lr_dec_min=MARCO_LR_DEC_MIN, lr_dec_max=MARCO_LR_DEC_MAX,
        patience=MARCO_PATIENCE,
        min_delta_pct=MARCO_MIN_DELTA_PCT,
        best_checkpoint_dir=BEST_CKPT_DIR,
        log_path=os.path.join(LOGS_DIR, f"marco_history_whisper_{DATA_SCALE_HOURS}h.json"),
        language=LANGUAGE, task=TASK,
        max_new_tokens=MAX_GEN_TOKENS,
        resume_from_log=RESUME_FROM_CHECKPOINT,
        verbose=True,
    )

    # Disk-space guard
    disk_guard = DiskSpaceGuardCallback(OUTPUT_DIR, min_free_gb=8.0, verbose=True)

    # Trainer with PRE-BUILT optimizer
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_set,
        callbacks=[marco_callback, disk_guard],
        optimizers=(optimizer, None),            # we provide optimizer; scheduler is auto-built from it
    )

    print(f"\nTrainer initialized.")
    print(f"  Run name:                 {run_name}")
    print(f"  Initial LR (encoder):     {initial_lr_enc:.3e}")
    print(f"  Initial LR (decoder):     {initial_lr_dec:.3e}")
    print(f"  Train samples:            {len(train_set)} (~{actual_train_hours:.2f}h)")
    print(f"  Per-device batch:         {PER_DEVICE_BATCH_SIZE}")
    print(f"  Grad accum:               {GRADIENT_ACCUMULATION_STEPS}")
    print(f"  Effective batch:          {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"  Epochs (max):             {NUM_TRAIN_EPOCHS} (Marco-ASR may stop earlier)")
    print(f"  Eval / LR adjust every:   {MARCO_EVAL_STEPS} steps on {MARCO_EVAL_SUBSET_SIZE} utt")
    print(f"  Save every:               {training_args.save_steps} steps (keep last {training_args.save_total_limit})")
else:
    print(f"MODE={MODE}: skipping Trainer setup.")


Building custom optimizer with named param groups...
  Encoder param tensors:  384  (23.59M params)
  Decoder param tensors:  640  (34.08M params)

Trainer initialized.
  Run name:                 whisper-lv3-darija-marco-10h-r32-lrEnc7.7e-06-lrDec7.7e-05
  Initial LR (encoder):     7.737e-06
  Initial LR (decoder):     7.737e-05
  Train samples:            7892 (~10.00h)
  Per-device batch:         4
  Grad accum:               4
  Effective batch:          16
  Epochs (max):             3 (Marco-ASR may stop earlier)
  Eval / LR adjust every:   100 steps on 200 utt
  Save every:               200 steps (keep last 3)


### 7.6 Train

Same auto-fallback infrastructure as the Qwen2-Audio v2 notebook:

1. **Pre-flight disk check** — abort if < 15 GB free.
2. **`get_latest_checkpoint`** — strict regex (`^checkpoint-(\d+)$`) so renamed `.corrupted_*` directories don't match.
3. **`is_valid_checkpoint`** — verifies `trainer_state.json` + `adapter_config.json` exist.
4. **`train_with_auto_fallback`** — retries up to 3 times with progressively older checkpoints, renaming corrupted ones aside.

If the run is interrupted (e.g. disk fill, pod restart), re-running this cell auto-resumes from the latest valid checkpoint. The Marco-ASR callback replays its history JSON to restore `best_wer` / `no_improve_count`.


In [21]:
# === CELL 20: Train with auto-fallback ===
if MODE == "fine_tune":
    # Pre-flight disk check
    free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
    print(f"[PRE-FLIGHT] free disk in {OUTPUT_DIR}: {free_gb:.1f} GB")
    if free_gb < 15:
        raise RuntimeError(
            f"Pre-flight failed: only {free_gb:.1f} GB free; need ≥ 15 GB for training."
        )

    _CKPT_RE = re.compile(r"^checkpoint-(\d+)$")

    def get_latest_checkpoint(output_dir):
        """Find the highest-numbered checkpoint-N directory. STRICT regex (no .corrupted_*)."""
        if not os.path.isdir(output_dir):
            return None
        candidates = []
        for d in os.listdir(output_dir):
            full = os.path.join(output_dir, d)
            if not os.path.isdir(full):
                continue
            m = _CKPT_RE.match(d)
            if m:
                candidates.append((int(m.group(1)), full))
        if not candidates:
            return None
        candidates.sort()
        return candidates[-1][1]

    def is_valid_checkpoint(ckpt_path):
        required = ["trainer_state.json", "adapter_config.json"]
        return all(os.path.exists(os.path.join(ckpt_path, f)) for f in required)

    def train_with_auto_fallback(trainer, output_dir, max_fallbacks=3):
        """Try resuming from latest checkpoint; on failure, rename and try earlier."""
        for attempt in range(max_fallbacks + 1):
            resume_path = get_latest_checkpoint(output_dir)
            if resume_path is None:
                print(f"[FALLBACK] no checkpoints found, starting from scratch")
                return trainer.train()
            if not is_valid_checkpoint(resume_path):
                print(f"[FALLBACK] {resume_path} incomplete; renaming and trying earlier")
                os.rename(resume_path, resume_path + f".corrupted_{int(time.time())}")
                continue
            print(f"[FALLBACK] resuming from {resume_path} (attempt {attempt+1}/{max_fallbacks+1})")
            try:
                return trainer.train(resume_from_checkpoint=resume_path)
            except (RuntimeError, FileNotFoundError, EOFError, KeyError) as e:
                print(f"[FALLBACK] resume failed: {type(e).__name__}: {e}")
                if attempt >= max_fallbacks:
                    raise
                os.rename(resume_path, resume_path + f".corrupted_{int(time.time())}")
        raise RuntimeError("train_with_auto_fallback exited without completing")

    print(f"\n>>> Starting Whisper-LV3 fine-tuning ({DATA_SCALE_HOURS}h, r={LORA_RANK}) <<<\n")
    train_t0 = time.time()
    train_result = train_with_auto_fallback(trainer, TRAINER_OUTPUT_DIR, max_fallbacks=3)
    train_total = time.time() - train_t0

    # Save final model
    final_dir = os.path.join(OUTPUT_DIR, f"final_whisper_{DATA_SCALE_HOURS}h")
    trainer.save_model(final_dir)
    processor.save_pretrained(final_dir)
    print(f"\nFinal model saved to: {final_dir}")
    print(f"Best WER checkpoint:  {BEST_CKPT_DIR}")
    print(f"  WER {marco_callback.best_wer*100:.2f}% at step {marco_callback.best_step}")

    # Cost log
    train_cost = {
        "model": MODEL_ID,
        "data_scale_hours": DATA_SCALE_HOURS,
        "actual_train_hours": actual_train_hours,
        "n_train_samples": len(train_set),
        "wall_clock_sec": train_total,
        "wall_clock_hours": train_total / 3600,
        "best_wer_periodic_eval": marco_callback.best_wer,
        "best_step": marco_callback.best_step,
        "global_step_at_end": trainer.state.global_step,
        "initial_lr_encoder": initial_lr_enc,
        "initial_lr_decoder": initial_lr_dec,
        "wer0": wer0,
        "marco_lr_enc_range": [MARCO_LR_ENC_MIN, MARCO_LR_ENC_MAX],
        "marco_lr_dec_range": [MARCO_LR_DEC_MIN, MARCO_LR_DEC_MAX],
    }
    with open(os.path.join(RESULTS_DIR, f"training_cost_whisper_{DATA_SCALE_HOURS}h.json"), "w") as f:
        json.dump(train_cost, f, indent=2)
    print(f"\nTraining cost: {train_total/3600:.2f} GPU-hours")
else:
    print(f"MODE={MODE}: skipping training")


[PRE-FLIGHT] free disk in /workspace/outputs: 208535.5 GB

>>> Starting Whisper-LV3 fine-tuning (10h, r=32) <<<

[FALLBACK] no checkpoints found, starting from scratch
[MARCO-ASR] training begins; eval every 100 steps on 200 utt (batch=4)
[MARCO-ASR] encoder LR range: [1.00e-07, 1.00e-05]
[MARCO-ASR] decoder LR range: [1.00e-06, 1.00e-04]
[DISK-GUARD] before training start: 208538.2 GB free


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
25,0.848900
50,0.711400
75,0.613600
100,0.603700
125,0.554700
150,0.561900
175,0.565500
200,0.541300
225,0.554600
250,0.491800


[MARCO-ASR step=  100] ↓ WER= 44.38%  CER= 18.26%  LR_enc=4.49e-06  LR_dec=4.49e-05  best= 44.38% (step 100)  no_improve=0/3  eval=197.1s
[MARCO-ASR step=  200] ↓ WER= 40.08%  CER= 16.41%  LR_enc=4.07e-06  LR_dec=4.07e-05  best= 40.08% (step 200)  no_improve=0/3  eval=238.5s
[DISK-GUARD] before save at step 200: 209054.0 GB free
[MARCO-ASR step=  300] ↓ WER= 38.87%  CER= 15.53%  LR_enc=3.95e-06  LR_dec=3.95e-05  best= 38.87% (step 300)  no_improve=0/3  eval=214.6s
[MARCO-ASR step=  400] ↓ WER= 37.45%  CER= 15.24%  LR_enc=3.81e-06  LR_dec=3.81e-05  best= 37.45% (step 400)  no_improve=0/3  eval=203.5s
[DISK-GUARD] before save at step 400: 209291.9 GB free
[MARCO-ASR step=  500] → WER= 37.02%  CER= 15.18%  LR_enc=3.77e-06  LR_dec=3.77e-05  best= 37.45% (step 400)  no_improve=1/3  eval=182.9s
[MARCO-ASR step=  600] ↓ WER= 36.42%  CER= 14.80%  LR_enc=3.71e-06  LR_dec=3.71e-05  best= 36.42% (step 600)  no_improve=0/3  eval=185.3s
[DISK-GUARD] before save at step 600: 209256.5 GB free
[MARCO-

## 8. Post-training evaluation

Three evaluations on the **best-WER** Marco-ASR checkpoint (loaded from `BEST_CKPT_DIR`):

1. **MoulSot test set** (in-distribution headline)
2. **Casablanca Morocco** (OOD generalization)

Each is partitioned into `full` / `code_switched` / `monolingual` with bootstrap CIs (1000 resamples, 95%). Results are saved as JSON to `RESULTS_DIR/` for paired-bootstrap comparison against Qwen2-Audio in Section 9.


In [22]:
# === CELL 21: Load best checkpoint and run final evaluation ===
if MODE in ("fine_tune", "evaluate_only"):
    # Determine which adapter to load
    if MODE == "evaluate_only":
        adapter_path = EVAL_ADAPTER_PATH
        print(f"MODE=evaluate_only: loading adapter from {adapter_path}")
        # In evaluate_only, the model may not yet be PEFT-wrapped
        if not isinstance(model, PeftModel):
            model = PeftModel.from_pretrained(model, adapter_path)
        else:
            model.load_adapter(adapter_path, "default")
            model.set_adapter("default")
        model.to(DEVICE).eval()
    else:
        adapter_path = BEST_CKPT_DIR
        if os.path.exists(adapter_path):
            print(f"Loading best Marco-ASR checkpoint from {adapter_path}")
            print(f"  (val WER {marco_callback.best_wer*100:.2f}% at step {marco_callback.best_step})")
            model.load_adapter(adapter_path, "default")
            model.set_adapter("default")
        else:
            print(f"WARNING: best checkpoint {adapter_path} not found; using current model state")

    model.eval()

    # MoulSot test (in-distribution)
    print("\n=== MoulSot test set (in-distribution) ===")
    moulsot_test_results = evaluate_with_partitioning(
        test_set,
        model=model, processor=processor,
        forced_decoder_ids=FORCED_DECODER_IDS,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        max_new_tokens=MAX_GEN_TOKENS,
        num_beams=1,
        do_bootstrap=True,
        verbose=True,
        eval_name=f"final_test_whisper_{DATA_SCALE_HOURS}h",
        save_details=True,
    )

    # Casablanca OOD
    casablanca_results = None
    if casablanca_ood is not None:
        print("\n=== Casablanca Morocco (OOD) ===")
        casablanca_results = evaluate_with_partitioning(
            casablanca_ood,
            model=model, processor=processor,
            forced_decoder_ids=FORCED_DECODER_IDS,
            eval_batch_size=MARCO_EVAL_BATCH_SIZE,
            max_new_tokens=MAX_GEN_TOKENS,
            num_beams=1,
            do_bootstrap=True,
            verbose=True,
            eval_name=f"final_casablanca_ood_whisper_{DATA_SCALE_HOURS}h",
            save_details=True,
        )
    else:
        print("\nCasablanca OOD skipped (data not loaded)")
else:
    print(f"MODE={MODE}: skipping post-training evaluation")


Loading best Marco-ASR checkpoint from /workspace/outputs/best_marco_asr_whisper_10h
  (val WER 34.35% at step 1000)

=== MoulSot test set (in-distribution) ===
[final_test_whisper_10h] running inference on 1962 samples (batch=4, beams=1)...
  saved final_test_whisper_10h_details.json + _summary.json

[final_test_whisper_10h] results:
  full            (n= 1962): WER  40.14% [ 39.29,  41.03]  CER  14.13% [ 13.62,  14.68]
  code_switched   (n=  194): WER  43.57% [ 41.33,  45.63]  CER  18.30% [ 17.01,  19.67]
  monolingual     (n= 1768): WER  39.64% [ 38.70,  40.53]  CER  13.51% [ 12.98,  14.05]
  RTF: 0.187 (inference / audio)

=== Casablanca Morocco (OOD) ===
[final_casablanca_ood_whisper_10h] running inference on 500 samples (batch=4, beams=1)...
  saved final_casablanca_ood_whisper_10h_details.json + _summary.json

[final_casablanca_ood_whisper_10h] results:
  full            (n=  500): WER  61.12% [ 59.24,  62.86]  CER  23.13% [ 22.09,  24.19]
  code_switched  : n=0
  monolingual   

## 9. Paired bootstrap: zero-shot vs fine-tuned

The same comparison protocol as Qwen2-Audio. We have two predictions for **each test utterance** (zero-shot and fine-tuned). The paired bootstrap resamples utterance indices with replacement and computes the **WER delta** (FT − ZS) on each resampled set; the percentile CI on that delta tells us whether the improvement is significant.

This cell requires both the **zero-shot details JSON** (from the standalone Whisper-LV3 zero-shot notebook) and the **fine-tuned details JSON** (saved by Section 8). If the ZS file isn't present, this cell prints what's needed and exits cleanly.


In [ ]:
# === CELL 22: Paired bootstrap (zero-shot vs fine-tuned) ===
if MODE in ("fine_tune", "evaluate_only") and 'moulsot_test_results' in dir():
    zs_details_path = os.path.join(RESULTS_DIR, f"zero_shot_test_whisper_details.json")
    ft_details_path = os.path.join(RESULTS_DIR, f"final_test_whisper_{DATA_SCALE_HOURS}h_details.json")

    if not os.path.exists(zs_details_path):
        print(f"Zero-shot details file not found at: {zs_details_path}")
        print(f"  (run the Whisper-LV3 zero-shot notebook first and copy its test-set details JSON here)")
        print(f"  expected schema: list of {{raw_ref, raw_pred, norm_ref, norm_pred, is_cs}} entries, same order as test_set")
    elif not os.path.exists(ft_details_path):
        print(f"Fine-tuned details file not found at: {ft_details_path}")
        print(f"  (re-run section 8 to generate it)")
    else:
        print(f"Running paired bootstrap on MoulSot test (ZS vs FT-{DATA_SCALE_HOURS}h)...")
        with open(zs_details_path) as f:
            zs_details = json.load(f)
        with open(ft_details_path) as f:
            ft_details = json.load(f)

        # Match by index (both should have the test set in the same order)
        n_min = min(len(zs_details), len(ft_details))
        if len(zs_details) != len(ft_details):
            print(f"  WARNING: ZS has {len(zs_details)} entries, FT has {len(ft_details)}; using first {n_min}")

        # Build triple: (ref, zs_pred, ft_pred), already normalized
        triples = []
        for i in range(n_min):
            zs = zs_details[i]
            ft = ft_details[i]
            ref = zs.get("norm_ref", "")
            if not ref:
                continue
            triples.append((ref, zs.get("norm_pred", ""), ft.get("norm_pred", "")))

        print(f"  paired triples (ref, zs, ft): {len(triples)}")

        def paired_bootstrap_delta(triples, n_resamples=BOOTSTRAP_RESAMPLES, ci=BOOTSTRAP_CI, seed=SEED):
            """Delta = WER(ft) − WER(zs). Negative = FT improves."""
            rng = np.random.RandomState(seed)
            n = len(triples)
            refs = [t[0] for t in triples]
            zs_preds = [t[1] for t in triples]
            ft_preds = [t[2] for t in triples]

            wer_zs = float(jiwer_wer(refs, zs_preds))
            wer_ft = float(jiwer_wer(refs, ft_preds))
            delta_point = wer_ft - wer_zs

            boot_deltas = []
            for _ in range(n_resamples):
                idx = rng.choice(n, size=n, replace=True)
                r = [refs[i] for i in idx]
                z = [zs_preds[i] for i in idx]
                f = [ft_preds[i] for i in idx]
                try:
                    boot_deltas.append(float(jiwer_wer(r, f)) - float(jiwer_wer(r, z)))
                except Exception:
                    continue
            alpha = 1 - ci
            lo = float(np.percentile(boot_deltas, 100 * alpha / 2))
            hi = float(np.percentile(boot_deltas, 100 * (1 - alpha / 2)))
            return {
                "wer_zero_shot": wer_zs,
                "wer_fine_tuned": wer_ft,
                "delta": delta_point,
                "delta_ci_low": lo,
                "delta_ci_high": hi,
                "n": n,
                "n_resamples": n_resamples,
                "significant_improvement": hi < 0,
            }

        result = paired_bootstrap_delta(triples)
        sig = "significant" if result["significant_improvement"] else "NOT significant"
        print(f"\n  WER (zero-shot):    {result['wer_zero_shot']*100:.2f}%")
        print(f"  WER (fine-tuned):   {result['wer_fine_tuned']*100:.2f}%")
        print(f"  Δ WER:              {result['delta']*100:+.2f}pp   95% CI [{result['delta_ci_low']*100:+.2f}, {result['delta_ci_high']*100:+.2f}]   ({sig})")

        # Save
        out_path = os.path.join(RESULTS_DIR, f"paired_bootstrap_whisper_{DATA_SCALE_HOURS}h.json")
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)
        print(f"  saved to {out_path}")
else:
    print(f"MODE={MODE}: skipping paired bootstrap")


## 10. Marco-ASR training curves

Four panels:

1. **WER vs step** with the best-so-far curve overlaid
2. **CER vs step**
3. **Encoder LR + Decoder LR** vs step on a log scale (Algorithm 2's dual-LR trajectory)
4. **WER vs wall-clock time** (for cost/quality estimation)

Read the log file directly so this cell works even after kernel restart.


In [ ]:
# === CELL 23: Plot Marco-ASR Algorithm 2 curves ===
def plot_marco_history_whisper(log_path, save_path=None, show=True):
    """Plot WER + CER + encoder/decoder LRs from Marco-ASR Algorithm 2 log."""
    if not os.path.exists(log_path):
        print(f"No history found at {log_path}")
        return
    with open(log_path, "r") as f:
        history = json.load(f)
    if not history:
        print("History is empty")
        return

    steps = [h["step"] for h in history]
    wers = [h["wer"] * 100 for h in history]
    cers = [h["cer"] * 100 for h in history]
    lr_enc = [h["lr_encoder"] for h in history]
    lr_dec = [h["lr_decoder"] for h in history]
    elapsed = [h["elapsed_sec"] / 60 for h in history]

    # Best-so-far curve
    best_wer_curve = []
    best_so_far = float("inf")
    for w in wers:
        if w < best_so_far:
            best_so_far = w
        best_wer_curve.append(best_so_far)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(
        f"Whisper-LV3 Marco-ASR Algorithm 2 training curves "
        f"({DATA_SCALE_HOURS}h, r={LORA_RANK}, {len(history)} evals)",
        fontsize=13, fontweight="bold",
    )

    # WER
    ax = axes[0, 0]
    ax.plot(steps, wers, "o-", color="#1f77b4", alpha=0.7, markersize=4, label="per-eval WER")
    ax.plot(steps, best_wer_curve, "-", color="#d62728", linewidth=2, label="best WER so far")
    best_w = min(wers)
    best_s = steps[wers.index(best_w)]
    ax.scatter([best_s], [best_w], color="#d62728", s=80, zorder=5,
               label=f"best: {best_w:.2f}% @ step {best_s}")
    ax.set_xlabel("training step")
    ax.set_ylabel("WER (%)")
    ax.set_title("WER vs step")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)

    # CER
    ax = axes[0, 1]
    ax.plot(steps, cers, "o-", color="#2ca02c", alpha=0.7, markersize=4)
    ax.set_xlabel("training step")
    ax.set_ylabel("CER (%)")
    ax.set_title("CER vs step")
    ax.grid(alpha=0.3)

    # LR: encoder + decoder
    ax = axes[1, 0]
    ax.plot(steps, lr_enc, "o-", color="#ff7f0e", alpha=0.85, markersize=4, label="encoder LR")
    ax.plot(steps, lr_dec, "s-", color="#9467bd", alpha=0.85, markersize=4, label="decoder LR")
    ax.set_xlabel("training step")
    ax.set_ylabel("learning rate")
    ax.set_yscale("log")
    ax.set_title("Marco-ASR Algorithm 2 adaptive LRs")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3, which="both")

    # Wall-clock
    ax = axes[1, 1]
    ax.plot(elapsed, wers, "o-", color="#8c564b", alpha=0.7, markersize=4)
    ax.set_xlabel("wall-clock elapsed (minutes)")
    ax.set_ylabel("WER (%)")
    ax.set_title("WER vs wall-clock")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved curves to {save_path}")
    if show:
        plt.show()
    plt.close()

    print(f"\n  Total evals:           {len(history)}")
    print(f"  Best WER:              {best_w:.2f}% at step {best_s}")
    print(f"  Latest WER:            {wers[-1]:.2f}% at step {steps[-1]}")
    print(f"  Latest encoder LR:     {lr_enc[-1]:.2e}")
    print(f"  Latest decoder LR:     {lr_dec[-1]:.2e}")
    print(f"  Total wall-clock:      {elapsed[-1]:.1f} min")


log_path = os.path.join(LOGS_DIR, f"marco_history_whisper_{DATA_SCALE_HOURS}h.json")
plot_path = os.path.join(RESULTS_DIR, f"marco_curves_whisper_{DATA_SCALE_HOURS}h.png")
plot_marco_history_whisper(log_path, save_path=plot_path, show=True)


## 11. Summary

This run produced the following artifacts:

| File | Contents |
|---|---|
| `marco_history_whisper_{H}h.json` | Per-eval WER + CER + encoder/decoder LR trajectory |
| `wer0_whisper_{H}h_summary.json` | Zero-shot WER₀ on val subset (initial-LR seed) |
| `final_test_whisper_{H}h_summary.json` | Post-training MoulSot test results with CIs (full/CS/mono) |
| `final_test_whisper_{H}h_details.json` | Per-utterance predictions for paired bootstrap |
| `final_casablanca_ood_whisper_{H}h_summary.json` | Casablanca OOD results with CIs |
| `marco_curves_whisper_{H}h.png` | Four-panel training curves |
| `training_cost_whisper_{H}h.json` | Wall-clock + best-step + initial LRs |
| `lora_targets_whisper_{H}h.json` | LoRA module audit (encoder/decoder counts) |
| `paired_bootstrap_whisper_{H}h.json` | (after ZS results present) ZS-vs-FT delta with CI |

### Next steps for Phase 1 comparison

After both Qwen2-Audio (10h, 20h) and Whisper-LV3 (10h, 20h) have completed:

1. Compute **paired bootstrap on MoulSot test set** between Qwen2-Audio-FT and Whisper-FT at each scale → tests whether one architecture significantly outperforms the other.
2. Compare **OOD degradation** (MoulSot → Casablanca) across the two architectures → tests robustness claims.
3. Compare **CS-vs-Mono partition WERs** → tests how each architecture handles code-switched inputs.
4. Compute **scaling slope** (10h → 20h) for both → tests data efficiency.

### Methodological notes for the defense

- Both architectures use **identical**: dataset, splits, seeds, normalization, CS detection, bootstrap protocol, partition definitions, eval cadence, LoRA hyperparams, effective batch, early-stopping criteria, OOD eval set.
- The Marco-ASR variant **differs by architectural necessity**: Algorithm 1 (single LR) for the LLM-based Qwen2-Audio, Algorithm 2 (encoder + decoder LRs) for the encoder-decoder Whisper. This is the principled choice per Ni et al. 2025; using Algorithm 1 on Whisper would force the encoder and decoder to share a LR despite their very different optimization needs, biasing the comparison.
- LR ranges for Algorithm 2 are an order of magnitude apart by design (encoder 1e-7–1e-5, decoder 1e-6–1e-4). This is also stated in the Marco-ASR paper as the recommended default for Whisper-class models.
